# 01 · RQ1 — which design factors matter?

> Reference notebook. Every number below is produced by the same functions that
> write the paper's tables (`paper_resources/scripts/`), so this is a readable
> view of the pipeline rather than a second implementation. Regenerate the
> `.tex` files with `python scripts/run_all.py`.


**Question:** how do the configured rank *r* and the expert count affect
multilingual performance, holding the rest of the pipeline fixed?

Backs: Table 2 (design factors), Table A1 (full rank matrices, omnibus tests).

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(p for p in Path.cwd().resolve().parents if (p / "paper_resources").is_dir())
sys.path.insert(0, str(REPO_ROOT / "paper_resources" / "scripts"))

import common as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
# 4 significant digits rather than 4 decimals, so that a p-value of 5.6e-15 stays
# readable next to an effect size of 0.94.
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

cfg = C.load_config()
BENCHMARKS = cfg["analysis"]["benchmarks"]
ALPHA = cfg["analysis"]["alpha"]
print(f"repo root : {REPO_ROOT}")
print(f"benchmarks: {BENCHMARKS}")
print(f"alpha     : {ALPHA}")

repo root : /data/upb/users/n/nikit/profiles/unix/cs/projects/moe-study-analysis
benchmarks: ['Belebele', 'FLORES', 'SIB200', 'Taxi1500']
alpha     : 0.05


## Balanced panels

Both sweeps cover different method sets at different levels, so before ranking
anything the panel is restricted to the architectures present at *every* level.
Without this the comparison would partly measure which methods happened to be
run where. `prepare_rank_panel` and `prepare_component_panel` do exactly that.

In [2]:
import tab1_rq1_design_factors as rq1

rank_data, rank_kinds = rq1.prepare_rank_panel(cfg)
component_data, component_kinds, component_runs = rq1.prepare_component_panel(cfg)

print("rank levels     :", cfg["analysis"]["rank_levels"])
print("architectures   :", rank_kinds)
print("rank panel rows :", len(rank_data))
print()
print("expert levels   :", cfg["analysis"]["component_levels"])
print("architectures   :", component_kinds)
print("expert panel rows:", len(component_data))

rank levels     : [2, 4, 8, 16, 32, 64]
architectures   : ['hmora', 'hydralora', 'lora', 'moelora', 'mola', 'mtllora']
rank panel rows : 1512

expert levels   : [2, 4, 8, 16, 32]
architectures   : ['hmora', 'hydralora', 'moelora', 'mola', 'movlora', 'mtllora']
expert panel rows: 1260


## Rank: average position per benchmark

Systems are ranked inside each matched evaluation case, then averaged. Lower is
better. The two FLORES directions are averaged before the four benchmark
families are given equal weight, so a benchmark with more languages cannot
dominate.

In [3]:
rank_by_bench, rank_macro, rank_tests = C.average_rank_summary(
    rank_data, "level", cfg["analysis"]["rank_levels"], BENCHMARKS
)
display(rank_by_bench.round(2))
print("\nmacro average position (lower is better)")
rank_macro.round(2).sort_values()

level,2,4,8,16,32,64
Belebele,2.76,2.85,2.41,3.01,4.94,5.03
FLORES,2,2.37,2.36,3.85,4.83,5.58
SIB200,2.89,3.41,3.32,3.63,3.6,4.15
Taxi1500,2.31,2.62,2.79,3.46,4.6,5.22



macro average position (lower is better)


level
2    2.49
8    2.72
4    2.81
16   3.49
32   4.49
64      5
dtype: float64

`r=2` takes the best macro position and `r=8` is second. Because adapter size
grows with rank, the winning setting is also the cheapest — the finding that
makes the rest of the study affordable.

In [4]:
display(rank_tests)
print("Kendall's W: how much the evaluation cases agree about the ordering.")
print("High on FLORES, near zero on SIB-200 - rank matters far more on some benchmarks than others.")

,benchmark,n_blocks,friedman_q,kendalls_w,p_raw,p_holm
0,Belebele,54,106.2,0.3935,2.553e-21,7.658e-21
1,FLORES,108,341.4,0.6323,1.222e-71,4.889e-71
2,SIB200,54,13.6,0.05038,0.01833,0.01833
3,Taxi1500,36,74.52,0.414,1.17e-14,2.34e-14


Kendall's W: how much the evaluation cases agree about the ordering.
High on FLORES, near zero on SIB-200 - rank matters far more on some benchmarks than others.


The omnibus tests establish only that the levels differ from each other. They do
**not** license "r=2 beats r=8": that pairwise claim is not tested here, and the
paper does not make it.

In [5]:
comp_by_bench, comp_macro, comp_tests = C.average_rank_summary(
    component_data, "level", cfg["analysis"]["component_levels"], BENCHMARKS
)
display(comp_by_bench.round(2))
print("\nmacro average position")
display(comp_macro.round(2).sort_values())
comp_tests

level,2,4,8,16,32
Belebele,3.16,2.8,3.76,2.81,2.47
FLORES,3.59,2.89,2.94,3.5,2.08
SIB200,3.4,2.67,3.27,2.19,3.48
Taxi1500,3.68,3,3.47,2.51,2.33



macro average position


level
32   2.59
16   2.75
4    2.84
8    3.36
2    3.46
dtype: float64

,benchmark,n_blocks,friedman_q,kendalls_w,p_raw,p_holm
0,Belebele,54,20.87,0.09663,0.0003356,0.0006712
1,FLORES,108,62.99,0.1458,6.835e-13,2.734e-12
2,SIB200,54,28.47,0.1318,1.002e-05,3.005e-05
3,Taxi1500,36,20.54,0.1426,0.0003907,0.0006712


## Adapter cost

Parameter counts back the "cheapest setting also wins" claim. They stay in the
audit CSV rather than the paper table, because the per-level spread is a range
rather than one number.

> **Caveat.** In the rank-sweep workbook the `trainable_params` column is correct
> for the analysed levels (r = 2–64) but wrong at each family's *top* level
> (r ≥ 128), where several architectures report fewer trainable parameters than
> the level below. Those rows sit outside `rank_levels` and so outside every
> table here. Anyone widening `rank_levels` in `config.yaml` would pick them up.

In [6]:
rank_params = C.parameter_summary(
    rank_data, C.load_run_params(C.results_path(cfg, "rank_sweep_xlsx")), "level"
)
rank_params

,trainable_millions_mean,trainable_millions_min,trainable_millions_max
level,,,
2,47.13,2.623,144.1
4,64.37,5.244,148.7
8,98.87,10.49,197.7
16,167.9,20.97,365.4
32,306.1,41.94,701
64,583.3,83.89,1372


## Robustness: does the ordering survive a different aggregation?

Average position is one way to pool. Collapsing architectures and normalizing
within each evaluation case is another, with different sensitivities. If the two
disagree, the finding is an artefact of the aggregation.

In [7]:
norm_by_bench, norm_macro = C.architecture_collapsed_normalized_summary(
    rank_data, "level", cfg["analysis"]["rank_levels"], BENCHMARKS
)
display(norm_by_bench.round(3))
norm_macro.round(3).sort_values(ascending=False)

level,2,4,8,16,32,64
score_family,,,,,,
Belebele,0.97,0.868,0.919,0.552,0.241,0.003
FLORES,1,0.988,0.978,0.915,0.77,0
SIB200,0.879,0.91,0.716,0.719,0.122,0.086
Taxi1500,0.99,0.855,0.83,0.61,0.379,0


level
2     0.96
4    0.905
8    0.861
16   0.699
32   0.378
64   0.022
dtype: float64